# Тема 11. MCP (Model Context Protocol)

Демонстраційний код для живого заняття.


### Робота з OpenAI API

In [ ]:
!pip install langchain langchain-core langchain-openai langsmith "mcp<2" httpx
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 kB 7.4 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-openai]


### Налаштування змінних середовища та API ключів

In [ ]:
import os
from getpass import getpass
def get_secret(name: str, required: bool = True) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    hint = "" if required else " (Enter — пропустити)"
    return getpass(f"Введіть {name}{hint}: ")

os.environ['OPENROUTER_API_KEY'] = get_secret("OPENROUTER_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = get_secret("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "neoversity-topic-11"

### MCP

Ініціалізація моделі

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY") ,
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini",
    temperature=0
)

Демо

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("SimpleDemoServer")
#Опису для моделі тут немає окремо. FastMCP бере його з docstring, а схему параметрів — з анотацій типів.


@mcp.tool() #Для calculate виходить JSON-схема: a і b — числа, обов'язкові; operation — рядок зі значенням за замовчуванням multiply.
def calculate(a: float, b: float, operation: str = "multiply") -> str:
    """Executes basic arithmetic operations: add, subtract, multiply, divide."""
    if operation == "add": return str(a + b)
    if operation == "subtract": return str(a - b)
    if operation == "multiply": return str(a * b)
    if operation == "divide": return str(a / b) if b != 0 else "Error: Division by zero"
    return "Unknown operation"

@mcp.tool()
def get_user_city_weather(city: str) -> str:
    """Retrieves weather details for a specified city."""
    db = {"kyiv": "+12°C, overcast", "lviv": "+10°C, rain", "kharkiv": "+14°C, sunny"}
    return db.get(city.strip().lower(), f"Weather data for {city} is unavailable")

@mcp.tool()
def get_city_population(city: str) -> str:
    """Returns the approximate population of a Ukrainian city."""
    db = {"kyiv": "about 3 million", "lviv": "about 720 thousand", "kharkiv": "about 1.4 million"}
    return db.get(city.strip().lower(), f"No data for {city}")

@mcp.resource("docs://cities")
def cities_list() -> str:
   """List of cities with weather data."""
   return "Kyiv, Lviv, Kharkiv"

@mcp.prompt()
def weather_report(city: str) -> str:
   """Template for a short weather report."""
   return f"Write a two-sentence weather report for {city} using get_user_city_weather."


if __name__ == "__main__":
    mcp.run(transport="sse")

Overwriting mcp_server.py


In [ ]:
# @title
import os, sys, subprocess, time
from mcp import ClientSession
from mcp.client.sse import sse_client

# Надійне очищення порту 8000 перед стартом
if sys.platform != "win32":
    os.system("fuser -k 8000/tcp > /dev/null 2>&1")
else:
    os.system("FOR /F \"tokens=5\" %a IN ('netstat -a -n -o ^| findstr :8000') DO taskkill /F /PID %a >nul 2>&1")

srv = subprocess.Popen([sys.executable, "mcp_server.py"], stderr=subprocess.PIPE, text=True)
time.sleep(3)

# Перевірка працездатності сервера
if srv.poll() is not None:
    _, stderr_output = srv.communicate()
    print(f"❌ [CRITICAL]: Server failed to start!\nError:\n{stderr_output}")
else:
    try:
        async with sse_client("http://localhost:8000/sse") as (r, w):
            async with ClientSession(r, w) as s:
                await s.initialize()
                print("Tools:", [t.name for t in (await s.list_tools()).tools])
                print("Resources:", [str(x.uri) for x in (await s.list_resources()).resources])
                print("Resource text:", (await s.read_resource("docs://cities")).contents[0].text)
                print("Prompts:", [p.name for p in (await s.list_prompts()).prompts])
                msg = await s.get_prompt("weather_report", {"city": "Lviv"})
                print("Prompt text:", msg.messages[0].content.text)
    except Exception as e:
        print(f"❌ [CONNECTION ERROR]: {e}")
    finally:
        srv.terminate()
        srv.wait()

Tools: ['calculate', 'get_user_city_weather', 'get_city_population']
Resources: ['docs://cities']
Resource text: Kyiv, Lviv, Kharkiv
Prompts: ['weather_report']
Prompt text: Write a two-sentence weather report for Lviv using get_user_city_weather.


In [ ]:
import sys
import os
import subprocess
import time
import asyncio
from IPython.display import display, HTML

# 0. Clear port 8000 to prevent port-in-use conflicts
if sys.platform != "win32":
    os.system("fuser -k 8000/tcp > /dev/null 2>&1")
else:
    os.system("FOR /F \"tokens=5\" %a IN ('netstat -a -n -o ^| findstr :8000') DO taskkill /F /PID %a >nul 2>&1")

display(HTML("<b>[1/3] Starting MCP server in background...</b>"))

# 1. Start the MCP server process
server_process = subprocess.Popen(
    [sys.executable, "mcp_server.py"],
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(3)
if server_process.poll() is not None:
    _, stderr_output = server_process.communicate()
    print(f"❌ [CRITICAL]: Server failed to start!\n{stderr_output}", flush=True)
else:
    print("[*] MCP server started successfully.", flush=True)

display(HTML("<b>[2/3] Connecting client and running agent...</b>"))

from mcp import ClientSession
from mcp.client.sse import sse_client
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from pydantic import create_model, Field

async def run_demo():
    sse_url = "http://localhost:8000/sse"

    if not os.environ.get("OPENAI_API_KEY") and not os.environ.get("OPENROUTER_API_KEY"):
        print("❌ [ERROR]: API key is missing!", flush=True)
        return

    global llm
    if 'llm' not in globals():
        llm = ChatOpenAI(model="gpt-4o", temperature=0)

    # Connect to the MCP server via SSE transport
    async with sse_client(sse_url) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Discover tools provided by the MCP server - виявлення інструментів. Клієнт не знав заздалегідь, що вміє сервер, і дізнався це через протокол.
            #У виводі: «Discovered 2 tool(s)» із назвами та описами з docstring. Під капотом це JSON-RPC запит tools/list, а виклик session.call_tool — це tools/cal
            tools_list = await session.list_tools()
            print(f"[*] Discovered {len(tools_list.tools)} tool(s) from MCP server:", flush=True)
            for t in tools_list.tools:
                print(f"    - {t.name}: {t.description}", flush=True)
            print("-" * 50, flush=True)
            # кожен MCP-інструмент перетворюється на звичний LangChain-інструмент (StructuredTool). Коли агент його викликає, обгортка робить session.call_tool(...).
            # Це місток між протоколом і фреймворком. Агент далі не знає, що інструменти живуть в іншому процесі.
            langchain_tools = []
            for t in tools_list.tools:
                t_name = t.name
                t_desc = t.description
                t_schema = t.inputSchema or {}
                props = t_schema.get("properties", {})

                fields = {
                    k: (float if v.get("type") in ["number", "integer"] else str, Field(..., description=v.get("description", "")))
                    for k, v in props.items()
                }
                ArgsSchema = create_model(f"{t_name}_schema", **fields)

                # Factory wrapper for async tool execution
                def make_async_invoker(s, name, keys):
                    async def async_invoker(**kwargs):
                        filtered_args = {k: v for k, v in kwargs.items() if k in keys}
                        result = await s.call_tool(name, arguments=filtered_args)
                        return result.content[0].text
                    return async_invoker

                async_tool_invoker = make_async_invoker(session, t_name, set(props.keys()))

                lc_tool = StructuredTool.from_function(
                    coroutine=async_tool_invoker,
                    name=t_name,
                    description=t_desc,
                    args_schema=ArgsSchema
                )
                langchain_tools.append(lc_tool)

            # Initialize LangGraph ReAct agent
            agent = create_react_agent(llm, langchain_tools)

            # query = "What is 125 multiplied by 8, and what is the current weather in Lviv?"
            query = "What is the population of Kharkiv and what is the weather there?"
            print(f"\n[User Query]: {query}\n" + "-"*50, flush=True)

            inputs = {"messages": [HumanMessage(content=query)]}

            # Execute agent asynchronously
            response = await agent.ainvoke(inputs)

            for msg in response["messages"]:
                # Print tool requests made by the agent
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"🤖 [Tool Requested]: {tc['name']} | Args: {tc['args']}", flush=True)
                # Print tool execution results
                elif msg.__class__.__name__ == "ToolMessage":
                    print(f"⚙️ [Tool Output]: {msg.content}", flush=True)
                # Print final assistant responses (checking type or content)
                elif msg.__class__.__name__ == "AIMessage" and msg.content:
                    print(f"\n💬 [Final Assistant Response]: {msg.content}", flush=True)

    print("\n[*] Session closed successfully.", flush=True)

try:
    await run_demo()
except BaseException as e:
    if "unhandled errors in a TaskGroup" not in str(e):
        print(f"\n[ERROR]: {e}", flush=True)
finally:
    # Clean up background server process
    server_process.terminate()
    server_process.wait()
    display(HTML("<b>[3/3] Server stopped. Demo finished!</b>"))

[*] MCP server started successfully.


[*] Discovered 3 tool(s) from MCP server:
    - calculate: Executes basic arithmetic operations: add, subtract, multiply, divide.
    - get_user_city_weather: Retrieves weather details for a specified city.
    - get_city_population: Returns the approximate population of a Ukrainian city.
--------------------------------------------------

[User Query]: What is the population of Kharkiv and what is the weather there?
--------------------------------------------------
🤖 [Tool Requested]: get_city_population | Args: {'city': 'Kharkiv'}
🤖 [Tool Requested]: get_user_city_weather | Args: {'city': 'Kharkiv'}
⚙️ [Tool Output]: about 1.4 million
⚙️ [Tool Output]: +14°C, sunny

💬 [Final Assistant Response]: The population of Kharkiv is about 1.4 million, and the current weather there is 14°C with sunny conditions.

[*] Session closed successfully.


Чи треба щоразу писати таку обгортку, як у колабі?
Ні. Для LangChain і LangGraph є готова бібліотека-адаптер (langchain-mcp-adapters), яка перетворює MCP-інструменти на LangChain-інструменти. Реалізація зроблена обгортку вручну, щоб було видно, що відбувається під капотом

